# Mechanistic Uncertainty-Causal CP — V2 Phases 0-2 (gated)

Self-contained. Closes the gap between the v1 spec and Run 5's simplified pipeline:
- **Phase 0** — layer sweep (spec §4.5): is difficulty encoded somewhere other than the pre-head layer? Cheap `max|corr(activation PC, S_t)|` across 7 taps, all datasets. Pre-registered GO threshold 0.30 (Run 5's pre-head ceiling was 0.09-0.22).
- **Phase 1** — dose-response `P_j`/`U_j` from one intervention, residual `R_j = U_j - E[U_j|P_j]`, the A/B/C/D feature-type table, and `MVDF_v2` selection (|U_j| gate + top-k by R_j). H1/H1b/H1c tables.
- **Phase 2** — cross-environment invariance `I(R)` (the measurement Run 5 skipped) for every feature-set condition, on E_eval_ID + shift sub-blocks, with the hard leakage assertion (E_select never re-touched).

**Kaggle:** GPU + Internet on; attach the `uom220001c` multivariate-TS bundle. Run All.

**Pre-registered outcomes (spec §7):** `I(MVDF_v2) < I(correlated)` across datasets → positive/row 1-2; `I(MVDF_v2) ≈ I(correlated)` → row 3, a genuine negative on the invariance hypothesis (distinct from Run 5, which never measured invariance).

## 1. Vendor the package

In [ ]:
import os, base64, json
PKG='mechanistic_conformal'
FILES=json.loads(base64.b64decode('...').decode())
for rel,content in FILES.items():
    p=os.path.join(PKG,rel); os.makedirs(os.path.dirname(p),exist_ok=True); open(p,'w').write(content)
print('wrote',len(FILES),'files')


## 2. Deps + probe inputs

In [ ]:
import importlib,subprocess,sys,os
for pkg in ['pyyaml','scipy','scikit-learn','statsmodels']:
    n={'pyyaml':'yaml','scikit-learn':'sklearn'}.get(pkg,pkg)
    try: importlib.import_module(n)
    except ImportError: subprocess.run([sys.executable,'-m','pip','install','-q',pkg],check=True)
import torch; print('torch',torch.__version__,'cuda',torch.cuda.is_available())
sys.path.insert(0,'mechanistic_conformal'); os.chdir('mechanistic_conformal')
from data.probe_inputs import probe; probe()


## 3. Phase 0 — layer sweep (gate)

In [ ]:
import runpy, sys, torch, json
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
import yaml
for cf in ['configs/v2_phase0.yaml','configs/v2_phase1.yaml','configs/v2_phase2.yaml']:
    c=yaml.safe_load(open(cf)); c['device']=dev; yaml.safe_dump(c,open(cf,'w'))
sys.argv=['p0','--config','configs/v2_phase0.yaml','--datasets',*["ETTh1", "ETTh2", "ETTm1", "ETTm2", "traffic", "electricity", "weather", "solar"],]
try: runpy.run_module('experiments.phase0_layer_sweep', run_name='__main__')
except SystemExit: pass
p0=json.load(open('runs/v2_phase0/layer_sweep_summary.json'))
print('\nPHASE 0:', p0['result'], '| winning:', p0['winning_layers'])


## 4. Phase 1 — dose-response, R_j, feature-type table, MVDF_v2

In [ ]:
import runpy, sys
sys.argv=['p1','--config','configs/v2_phase1.yaml','--datasets',*["ETTh1", "ETTh2", "ETTm1", "ETTm2", "traffic", "electricity", "weather", "solar"],]
try: runpy.run_module('experiments.phase1_core_mechanism', run_name='__main__')
except SystemExit: pass
import pandas as pd
print('\n=== H1c enrichment (type-B) across runs ===')
display(pd.read_csv('runs/v2_phase1/AGG_h1c.csv'))


## 5. Phase 2 — cross-environment invariance (flagship)

In [ ]:
import runpy, sys
sys.argv=['p2','--config','configs/v2_phase2.yaml','--datasets',*["ETTh1", "ETTh2", "ETTm1", "ETTm2", "traffic", "electricity", "weather", "solar"],]
try: runpy.run_module('experiments.phase2_invariance', run_name='__main__')
except SystemExit: pass
import pandas as pd
agg=pd.read_csv('runs/v2_phase2/AGG_invariance.csv')
print('=== AGG_invariance.csv ==='); display(agg)
cols=[c for c in agg.columns if c.startswith('I_') and c not in ('I_MVDFv2_lt_correlated','I_MVDFv2_lt_MVDFv1')]
print('\nMean I by condition (lower = more invariant):')
print(agg[cols].mean().round(4).to_string())
print('\nI(MVDF_v2) < I(correlated):', agg['I_MVDFv2_lt_correlated'].tolist())
print('I(MVDF_v2) < I(MVDF_v1)  :', agg['I_MVDFv2_lt_MVDFv1'].tolist())
n=agg['I_MVDFv2_lt_correlated'].sum(); tot=agg['I_MVDFv2_lt_correlated'].notna().sum()
print(f'\nMVDF_v2 more invariant than correlated on {n}/{tot} (dataset,tap) runs')


## 6. Zip

In [ ]:
import shutil; shutil.make_archive('/kaggle/working/v2_runs','zip','runs')
print('runs -> /kaggle/working/v2_runs.zip')
